# Latency Testing Notebook

This notebook compares inference latency across different model architectures:

1. **TFM (Temporal Feature Model)** - Single-session basis function model
   - Uncompiled on GPU
   - Uncompiled on CPU
   - Compiled on CPU

2. **LSTM** - Recurrent baseline model
   - GPU
   - CPU

3. **Multisession Model** - Full production inference pipeline
   - Normalizers (z-scoring)
   - Autoencoder (encode → latent space)
   - TBFM (prediction in latent space)
   - Autoencoder (decode → neural space)
   - Tested on both CPU and GPU

The multisession model represents the complete architecture needed for real-world deployment, including dimensionality reduction and normalization.

In [1]:
%load_ext autoreload
%autoreload 2

import os

import torch

torch.set_grad_enabled(False)

from tbfm import dataset as ds
from tbfm import multisession
# import mmodel
# import temporal_feature_model as tfm
import timings

# Set data directory
DATA_DIR = os.getenv("TBFM_DATA_DIR", "/var/data/opto-coproc/")

SESSION_ID = "MonkeyJ_20160426_Session2_S1"

# Load data - use unpack_stiminds=False to avoid dataset.py bug
# The timing wrapper will handle unpacking
dset, _ = multisession.load_stim_batched(
    window_size=184,
    session_subdir="torchraw",
    data_dir=DATA_DIR,
    unpack_stiminds=False,  # Keep packed, timing wrapper will unpack
    held_in_session_ids=[SESSION_ID],
    batch_size=7400,
    num_held_out_sessions=0,
)

batch = next(iter(dset))

# Extract data from the batch dictionary
# batch format: {session_id: (runway, stiminds, y)}
session_data = batch[SESSION_ID]
runway = session_data[0]
stimind = session_data[1]  # (batch, time, stimdim) - packed format
y = session_data[2]

print(f"Data shapes:")
print(f"  runway: {runway.shape}")
print(f"  stimind: {stimind.shape}")
print(f"  y: {y.shape}")

in_dim = y.shape[-1]
stimdim = stimind.shape[-1]
RUNWAY = runway.shape[1]
WINDOW_SIZE = RUNWAY + y.shape[1]
trial_len = WINDOW_SIZE - RUNWAY

print(f"\nDerived parameters:")
print(f"  in_dim (neural): {in_dim}")
print(f"  stimdim: {stimdim}")
print(f"  runway: {RUNWAY}")
print(f"  trial_len: {trial_len}")

# Get the dataset for timing tests  
data = dset.get_first_dset()

Data shapes:
  runway: torch.Size([7400, 20, 77])
  stimind: torch.Size([7400, 164, 3])
  y: torch.Size([7400, 164, 77])

Derived parameters:
  in_dim (neural): 77
  stimdim: 3
  runway: 20
  trial_len: 164


In [2]:
num_bases = 15
latent_dim = 2
basis_depth = 2

model = tfm.TFM(in_dim, stimdim, RUNWAY, num_bases, trial_len, batchy=y, latent_dim=latent_dim,
                            basis_depth=basis_depth, device="cuda:0")
model.load_state_dict(torch.load("stmodel.torch"))

model = timings.TimingModuleTFMUncompiled(model, RUNWAY)

elapsed = timings.time_inference(model, data, SESSION_ID)

mean_elapsed = sum(elapsed) / len(elapsed)
tfm_gpu_uncompiled_ms = mean_elapsed / (1000 * 1000)
print(f"Mean latency: {tfm_gpu_uncompiled_ms:.3f}ms")

NameError: name 'tfm' is not defined

In [ ]:
num_bases = 15
latent_dim = 2
basis_depth = 2

model = tfm.TFM(in_dim, stimdim, RUNWAY, num_bases, trial_len, batchy=y, latent_dim=latent_dim,
                            basis_depth=basis_depth, device="cpu")
model.load_state_dict(torch.load("stmodel.torch"))

model = timings.TimingModuleTFMUncompiled(model, RUNWAY)

elapsed = timings.time_inference(model, data, SESSION_ID)

mean_elapsed = sum(elapsed) / len(elapsed)
tfm_cpu_uncompiled_ms = mean_elapsed / (1000 * 1000)
print(f"Mean latency: {tfm_cpu_uncompiled_ms:.3f}ms")

Mean latency: 0.1751551384ms


In [ ]:
num_bases = 15
latent_dim = 2
basis_depth = 2

model = tfm.TFM(in_dim, stimdim, RUNWAY, num_bases, trial_len, batchy=y, latent_dim=latent_dim,
                            basis_depth=basis_depth, device="cpu")
model.load_state_dict(torch.load("stmodel.torch"))

model = model.compile(stimind[:1, RUNWAY:, :])

model = timings.TimingModuleTFMCompiled(model, RUNWAY)

elapsed = timings.time_inference(model, data, SESSION_ID)

mean_elapsed = sum(elapsed) / len(elapsed)
tfm_cpu_compiled_ms = mean_elapsed / (1000 * 1000)
print(f"Mean latency: {tfm_cpu_compiled_ms:.3f}ms")

Mean latency: 0.1258873076ms


In [ ]:
lstm_model_path = os.path.join("models", "lstm", "a67cec30-3bc5-11ef-8d71-08bfb8868f6a")
model = mmodel.get(                                                                                           
    batch,                                                                                              
    latent_dim=96,
    weight_path=lstm_model_path,
    device="cuda:0",                                                                                            
)
model = timings.TimingModuleLSTM(model, RUNWAY)

elapsed = timings.time_inference(model, data, SESSION_ID)

mean_elapsed = sum(elapsed) / len(elapsed)
lstm_gpu_ms = mean_elapsed / (1000 * 1000)
print(f"Mean latency: {lstm_gpu_ms:.3f}ms")

Mean latency: 32.234226974ms


In [ ]:
lstm_model_path = os.path.join("models", "lstm", "a67cec30-3bc5-11ef-8d71-08bfb8868f6a")
model = mmodel.get(                                                                                           
    batch,                                                                                              
    latent_dim=96,
    weight_path=lstm_model_path,
    device="cpu",                                                                                            
)
model = timings.TimingModuleLSTM(model, RUNWAY)

elapsed = timings.time_inference(model, data, SESSION_ID)

mean_elapsed = sum(elapsed) / len(elapsed)
lstm_cpu_ms = mean_elapsed / (1000 * 1000)
print(f"Mean latency: {lstm_cpu_ms:.3f}ms")

Mean latency: 37.6904349502ms


# Multisession Model Timing Tests

Test the full multisession architecture (normalizers + autoencoder + TBFM) on both CPU and GPU.

In [6]:
%load_ext autoreload
%autoreload 2

import os

import torch

torch.set_grad_enabled(False)
# Load a trained multisession model from test folder
import tbfm.multisession as multisession

# Choose a model directory - update this path to your trained model
MODEL_DIR = "/home/danmuir/GitHub/py-tbfm/session_count_sweep_20251203_222740/1kx25_maml"

# Load model components
model_state = torch.load(os.path.join(MODEL_DIR, "model.torch"), map_location="cpu")
held_in_session_ids = torch.load(os.path.join(MODEL_DIR, "hisi.torch"))
embeddings_stim = torch.load(os.path.join(MODEL_DIR, "es.torch"), map_location="cpu")
hyperparams = torch.load(os.path.join(MODEL_DIR, "hyperparameters.torch"))

print(f"Loaded model with hyperparameters:")
for k, v in hyperparams.items():
    print(f"  {k}: {v}")

# Load rest embeddings
embeddings_rest = multisession.load_rest_embeddings(held_in_session_ids, device="cpu")

# Need to rebuild model from config - load the config
from pathlib import Path
from hydra import initialize_config_dir, compose

conf_dir = Path("./conf").resolve()
with initialize_config_dir(config_dir=str(conf_dir), version_base=None):
    cfg = compose(config_name="config")

print(f"\nTest session: {SESSION_ID}")
print(f"Embeddings rest shape for session: {embeddings_rest[SESSION_ID].shape}")
print(f"Embeddings stim shape for session: {embeddings_stim[SESSION_ID].shape}")
print(f"Stimind shape: {stimind.shape}, unpacked would be: {stimind[:, RUNWAY, :].shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded model with hyperparameters:
  num_bases: 100
  num_sessions: 25
  latent_dim: 96
  basis_residual_rank: 16
  residual_mlp_hidden: 16
  embed_dim_stim: 15
  embed_dim_rest: 3
  is_basis_residual: True
  coadapt: False
  train_size: 1000
  shuffle: True
  batch_size_per_session: 500
  use_two_stage: False

Test session: MonkeyJ_20160426_Session2_S1
Embeddings rest shape for session: torch.Size([3])
Embeddings stim shape for session: torch.Size([15])
Stimind shape: torch.Size([7400, 164, 3]), unpacked would be: torch.Size([7400, 3])


In [7]:
# Rebuild the multisession model exactly as tma_standalone.py does
# Set all the same config parameters that tma uses

# Core model dimensions
cfg.latent_dim = hyperparams['latent_dim']
cfg.tbfm.module.num_bases = hyperparams['num_bases']
cfg.tbfm.module.embed_dim_rest = hyperparams['embed_dim_rest']
cfg.tbfm.module.embed_dim_stim = hyperparams['embed_dim_stim']

# Training parameters (from tma_standalone.py lines 177-189)
cfg.training.epochs = 12001
cfg.ae.training.lambda_ae_recon = 0.03
cfg.ae.use_two_stage = hyperparams['use_two_stage']
cfg.ae.two_stage.freeze_only_shared = False
cfg.ae.two_stage.lambda_mu = 0.01
cfg.ae.two_stage.lambda_cov = 0.01
cfg.tbfm.training.lambda_fro = 75.0

# Meta-learning configuration
cfg.meta.is_basis_residual = hyperparams['is_basis_residual']
if hyperparams['is_basis_residual']:
    cfg.meta.basis_residual_rank = hyperparams['basis_residual_rank']
    cfg.meta.training.lambda_l2 = 1e-2
    cfg.meta.residual_mlp_hidden = hyperparams['residual_mlp_hidden']

cfg.meta.training.coadapt = hyperparams['coadapt']

# Override covariate_dim to match the trained model.
# bases.in_layer.weight has shape [basis_residual_rank, in_dim] where
# in_dim = covariate_dim + embed_dim_rest - 1 (clock vec removed in Bases.__init__)
# Checkpoint: [16, 5] => covariate_dim = 5 - 3 + 1 = 3
cfg.covariate_dim = 3
cfg.tbfm.module.covariate_dim = 3

print("Config before building:")
print(f"  covariate_dim: {cfg.covariate_dim}")
print(f"  tbfm.module.covariate_dim: {cfg.tbfm.module.covariate_dim}")
print(f"  embed_dim_rest: {cfg.tbfm.module.embed_dim_rest}")
print(f"  embed_dim_stim: {cfg.tbfm.module.embed_dim_stim}")
print(f"  is_basis_residual: {cfg.meta.is_basis_residual}")
print(f"  Expected bases input dim: {cfg.covariate_dim} + {cfg.tbfm.module.embed_dim_rest} - 1 = {cfg.covariate_dim + cfg.tbfm.module.embed_dim_rest - 1}")

# Create a minimal dummy batch with correct dimensions for model building
# Use a single sample with the right shapes
dummy_runway = runway[:1, :, :]  # (1, 20, in_dim)
dummy_stimind = torch.randn(1, 3)  # (1, 3) - unpacked stiminds with stimdim=3
dummy_y = y[:1, :, :]  # (1, 164, in_dim)

dummy_batch = {SESSION_ID: (dummy_runway, dummy_stimind, dummy_y)}

class DummyData:
    def __init__(self, batch):
        self._batch = batch
        self.session_ids = list(batch.keys())
    def keys(self):
        return self._batch.keys()
    def __iter__(self):
        yield self._batch
    def get_session_num_feats(self, session_id):
        return self._batch[session_id][0].shape[-1]

dummy_data = DummyData(dummy_batch)

# Build model using the dummy data
ms_model_cpu = multisession.build_from_cfg(cfg, dummy_data, device="cpu", quiet=False)

# Load the trained weights
ms_model_cpu.model.instances[SESSION_ID].load_state_dict(model_state)

print("\nMultisession model loaded successfully on CPU")
print(f"Model configuration:")
print(f"  latent_dim: {cfg.latent_dim}")
print(f"  num_bases: {cfg.tbfm.module.num_bases}")
print(f"  embed_dim_rest: {cfg.tbfm.module.embed_dim_rest}")
print(f"  embed_dim_stim: {cfg.tbfm.module.embed_dim_stim}")
print(f"  covariate_dim: {cfg.covariate_dim}")

Config before building:
  covariate_dim: 3
  tbfm.module.covariate_dim: 3
  embed_dim_rest: 3
  embed_dim_stim: 15
  is_basis_residual: True
  Expected bases input dim: 3 + 3 - 1 = 5
Building and fitting normalizers...
Building and warm starting AEs...
Building TBFM...
BOOM! Dino DNA!

Multisession model loaded successfully on CPU
Model configuration:
  latent_dim: 96
  num_bases: 100
  embed_dim_rest: 3
  embed_dim_stim: 15
  covariate_dim: 3


In [ ]:
# Time multisession model on CPU
ms_model_cpu.eval()

# Create a simple dataloader-like object for timing
# time_inference expects an iterable that yields (x, stiminds) tuples
class SimpleDataLoader:
    def __init__(self, x, stiminds):
        self.x = x
        self.stiminds = stiminds
    
    def __iter__(self):
        yield (self.x, self.stiminds)

# Concatenate runway and y to get full x
full_x = torch.cat([runway, y], dim=1)
timing_data = SimpleDataLoader(full_x, stimind)

timing_wrapper = timings.TimingModuleMultisession(
    ms_model_cpu, 
    RUNWAY,
    embeddings_rest,
    embeddings_stim,
    stimdim=2  # Training uses unpack_stiminds=True which gives dim 2 (no clock vec)
)

elapsed = timings.time_inference(timing_wrapper, timing_data, SESSION_ID, iterations=1000)

mean_elapsed = sum(elapsed) / len(elapsed)
multisession_cpu_ms = mean_elapsed / (1000 * 1000)
print(f"Multisession Model CPU - Mean latency: {multisession_cpu_ms:.3f}ms")

In [ ]:
# Now test on GPU
ms_model_gpu = multisession.build_from_cfg(cfg, dset, device="cuda:0")
ms_model_gpu.model.instances[SESSION_ID].load_state_dict(model_state)
ms_model_gpu.eval()

# Create a simple dataloader-like object for timing
class SimpleDataLoader:
    def __init__(self, x, stiminds):
        self.x = x
        self.stiminds = stiminds
    
    def __iter__(self):
        yield (self.x, self.stiminds)

# Concatenate runway and y to get full x
full_x = torch.cat([runway, y], dim=1)
timing_data_gpu = SimpleDataLoader(full_x, stimind)

# Move embeddings to GPU
embeddings_rest_gpu = {k: v.to("cuda:0") for k, v in embeddings_rest.items()}
embeddings_stim_gpu = {k: v.to("cuda:0") for k, v in embeddings_stim.items()}

timing_wrapper_gpu = timings.TimingModuleMultisession(
    ms_model_gpu, 
    RUNWAY,
    embeddings_rest_gpu,
    embeddings_stim_gpu,
    stimdim=2  # Training uses unpack_stiminds=True which gives dim 2 (no clock vec)
)

elapsed = timings.time_inference(timing_wrapper_gpu, timing_data_gpu, SESSION_ID, iterations=1000)

mean_elapsed = sum(elapsed) / len(elapsed)
multisession_gpu_ms = mean_elapsed / (1000 * 1000)
print(f"Multisession Model GPU - Mean latency: {multisession_gpu_ms:.3f}ms")

# Comparison Summary

Compare all models tested above to see the latency differences between:
- TFM (uncompiled) on CPU/GPU
- TFM (compiled) on CPU
- LSTM on CPU/GPU  
- **Multisession (AE + TBFM)** on CPU/GPU

The multisession model includes:
1. Normalizers (z-scoring)
2. Autoencoder encoding
3. TBFM prediction in latent space
4. Autoencoder decoding

This represents the full inference pipeline needed for real-world deployment.

In [ ]:
# Create a comparison table
# This cell automatically pulls results from the cells above

import pandas as pd

# Helper function to format results
def get_result(var_name):
    try:
        return f"{eval(var_name):.3f}"
    except:
        return "Not run yet"

# Collect results from variables set in cells above
results = {
    'Model': [
        'TFM (uncompiled, GPU)',
        'TFM (uncompiled, CPU)', 
        'TFM (compiled, CPU)',
        'LSTM (GPU)',
        'LSTM (CPU)',
        'Multisession (CPU)',
        'Multisession (GPU)',
    ],
    'Mean Latency (ms)': [
        get_result('tfm_gpu_uncompiled_ms'),
        get_result('tfm_cpu_uncompiled_ms'),
        get_result('tfm_cpu_compiled_ms'),
        get_result('lstm_gpu_ms'),
        get_result('lstm_cpu_ms'),
        get_result('multisession_cpu_ms'),
        get_result('multisession_gpu_ms'),
    ],
    'Components': [
        'TBFM only',
        'TBFM only',
        'TBFM only (compiled)',
        'LSTM only',
        'LSTM only',
        'Normalizer + AE + TBFM',
        'Normalizer + AE + TBFM',
    ]
}

df = pd.DataFrame(results)
print("\n" + "="*70)
print("LATENCY COMPARISON")
print("="*70)
print(df.to_string(index=False))
print("="*70)
print(f"\nTest configuration:")
print(f"  Session: {SESSION_ID}")
print(f"  Window size: {WINDOW_SIZE}")
print(f"  Runway: {RUNWAY}")
print(f"  Forecast horizon: {WINDOW_SIZE - RUNWAY}")
print(f"  Iterations: 1000 (10000 for single models)")
print(f"  Neural dimensionality: {in_dim}")